In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/airbnb.csv") 

df = df.copy()
df.head()

,id,host id,host_identity_verified,neighbourhood group,neighbourhood,lat,long,country,instant_bookable,cancellation_policy,room type,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365
0,1001254,80014485718,unconfirmed,Brooklyn,Kensington,40.64749,-73.97237,United States,False,strict,Private room,966.0,193.0,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0
1,1002102,52335172823,verified,Manhattan,Midtown,40.75362,-73.98377,United States,False,moderate,Entire home/apt,142.0,28.0,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0
2,1002403,78829239556,NaN,Manhattan,Harlem,40.80902,-73.94190,United States,True,flexible,Private room,620.0,124.0,3.0,0.0,NaN,0.00,5.0,1.0,352.0
3,1002755,85098326012,unconfirmed,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,True,moderate,Entire home/apt,368.0,74.0,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0
4,1003689,92037596077,verified,Manhattan,East Harlem,40.79851,-73.94399,United States,False,moderate,Entire home/apt,204.0,41.0,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0


In [15]:
df.shape

(98970, 20)

Target

In [16]:
df["demand_score"] = df["reviews per month"] / (df["availability 365"] + 1)

Demand Features

In [17]:
# review intensity
df["review_intensity"] = df["reviews per month"] / (df["number of reviews"] + 1)

# availability bucket
def availability_bucket(x):
    if x <= 120:
        return "low"
    elif x <= 240:
        return "medium"
    else:
        return "high"

df["availability_bucket"] = df["availability 365"].apply(availability_bucket)

Price Features

In [18]:
df["price_per_minimum_night"] = df["price"] / df["minimum nights"]

df["price_log"] = np.log1p(df["price"]) 

Location Features

In [19]:
neighbourhood_demand = (
    df.groupby("neighbourhood group")["demand_score"]
    .mean()
    .to_dict()
)

df["neighbourhood_group_demand_avg"] = df["neighbourhood group"].map(neighbourhood_demand)

Interation Features

In [20]:
df["room_type_neighbourhood"] = (
    df["room type"] + "_" + df["neighbourhood group"]
)

df["price_review_interaction"] = df["price"] * df["reviews per month"]

df["availability_min_nights"] = (
    df["availability 365"] * df["minimum nights"]
)

In [21]:
df.head()

,id,host id,host_identity_verified,neighbourhood group,neighbourhood,lat,long,country,instant_bookable,cancellation_policy,...,availability 365,demand_score,review_intensity,availability_bucket,price_per_minimum_night,price_log,neighbourhood_group_demand_avg,room_type_neighbourhood,price_review_interaction,availability_min_nights
0,1001254,80014485718,unconfirmed,Brooklyn,Kensington,40.64749,-73.97237,United States,False,strict,...,286.0,0.000732,0.021000,high,96.600000,6.874198,0.141970,Private room_Brooklyn,202.86,2860.0
1,1002102,52335172823,verified,Manhattan,Midtown,40.75362,-73.98377,United States,False,moderate,...,228.0,0.001659,0.008261,medium,4.733333,4.962845,0.133833,Entire home/apt_Manhattan,53.96,6840.0
2,1002403,78829239556,NaN,Manhattan,Harlem,40.80902,-73.94190,United States,True,flexible,...,352.0,0.000000,0.000000,high,206.666667,6.431331,0.133833,Private room_Manhattan,0.00,1056.0
3,1002755,85098326012,unconfirmed,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,True,moderate,...,322.0,0.014365,0.017122,high,12.266667,5.910797,0.141970,Entire home/apt_Brooklyn,1707.52,9660.0
4,1003689,92037596077,verified,Manhattan,East Harlem,40.79851,-73.94399,United States,False,moderate,...,289.0,0.000345,0.010000,high,20.400000,5.323010,0.133833,Entire home/apt_Manhattan,20.40,2890.0


Save

In [22]:
df.to_csv("../data/processed/feature_engineered_data.csv", index=False)